# Практична робота №4. Чищення набору даних
**Дисципліна:** «Аналіз даних та математичне моделювання»  
**Студент:** Хрустовський Павло  
**Група:** ІТ-42  
**Варіант:** 2  

### Параметри варіанта 2:
- **Місто:** Львів
- **Базова ціна:** 720 грн
- **Базова кількість:** 4

---
### Мета роботи:
Побудувати "брудний" набір даних із типовими дефектами (пропуски, дублікати, неправильні типи, неузгоджені категорії, викид) і послідовно усунути кожен дефект засобами бібліотеки `pandas` згідно з алгоритмом чищення даних.

In [1]:
import numpy as np
import pandas as pd

# Демонстраційний набір пацієнтів
patients = pd.DataFrame([
    {"id": 1, "вік": 34, "вага": "70 кг", "група_крові": "A+"},
    {"id": 2, "вік": np.nan, "вага": "82 кг", "група_крові": "o-"},
    {"id": 3, "вік": 61, "вага": "65 кг", "група_крові": "B+ "},
    {"id": 3, "вік": 61, "вага": "65 кг", "група_крові": "B+ "}, # дублікат id=3
    {"id": 4, "вік": 29, "вага": "58 кг", "група_крові": "AB+"},
    {"id": 5, "вік": 250, "вага": "77 кг", "група_крові": "a+"}, # 250 років — очевидний викид
])

print("Початковий демонстраційний датасет:")
display(patients)

# 1. Пропуски
print("\n1. Кількість пропусків у 'вік':", patients["вік"].isna().sum())
patients["вік"] = patients["вік"].fillna(patients["вік"].median())

# 2. Дублікати
print("2. Кількість дублікатів за 'id':", patients.duplicated(subset=["id"]).sum())
patients = patients.drop_duplicates(subset=["id"]).reset_index(drop=True)

# 3. Типи даних
patients["вага"] = patients["вага"].str.replace(" кг", "", regex=False).astype(float)

# 4. Узгодження категорій
patients["група_крові"] = patients["група_крові"].str.strip().str.upper()

# 5. Викиди (IQR)
q1, q3 = patients["вік"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"5. Межі IQR для віку: [{lower:.1f}, {upper:.1f}]")
outliers_demo = patients[(patients["вік"] < lower) | (patients["вік"] > upper)]
print("Виявлені викиди у віці:")
display(outliers_demo)
print("\nОчищений демонстраційний датасет:")
display(patients)

Початковий демонстраційний датасет:


,id,вік,вага,група_крові
0,1,34.0,70 кг,A+
1,2,NaN,82 кг,o-
2,3,61.0,65 кг,B+
3,3,61.0,65 кг,B+
4,4,29.0,58 кг,AB+
5,5,250.0,77 кг,a+



1. Кількість пропусків у 'вік': 1
2. Кількість дублікатів за 'id': 1
5. Межі IQR для віку: [-6.5, 101.5]
Виявлені викиди у віці:


,id,вік,вага,група_крові
4,5,250.0,77.0,A+



Очищений демонстраційний датасет:


,id,вік,вага,група_крові
0,1,34.0,70.0,A+
1,2,61.0,82.0,O-
2,3,61.0,65.0,B+
3,4,29.0,58.0,AB+
4,5,250.0,77.0,A+


---
## Завдання 1. Побудова брудного набору (Варіант 2: Львів)
Побудувати DataFrame із 8 замовлень для товару свого варіанта (базова ціна = 720 грн, базова кількість = 4), в якому явно присутні всі 5 дефектів:
1. 2 рядки з пропущеним значенням (`NaN`) у стовпці `"кількість"`;
2. 1 рядок, що повністю повторює інший (дублікат);
3. Щонайменше 2 значення в стовпці `"ціна"` записані як рядок із суфіксом `" грн"`, а не число;
4. Назва міста записана різними варіантами (звичайний `"Львів"`, з пробілами навколо `" Львів "`, великими літерами `"ЛЬВІВ"`, латиницею `"Lviv"`);
5. Одне значення `"ціна"` в 10 разів перевищує базову ціну (7200 грн замість 720 грн).

In [2]:
# Побудова брудного набору даних для Варіанта 2 (Львів)
raw_orders = pd.DataFrame([
    {"id": 1, "товар": "Кавоварка", "місто": "Львів", "ціна": 720, "кількість": 4},
    {"id": 2, "товар": "Кавоварка", "місто": " Львів ", "ціна": "720 грн", "кількість": np.nan},
    {"id": 3, "товар": "Кавоварка", "місто": "ЛЬВІВ", "ціна": 720, "кількість": 4},
    {"id": 4, "товар": "Кавоварка", "місто": "Львів", "ціна": "720 грн", "кількість": np.nan},
    {"id": 5, "товар": "Кавоварка", "місто": "Lviv", "ціна": 7200, "кількість": 4},
    {"id": 6, "товар": "Кавоварка", "місто": " Львів", "ціна": 720, "кількість": 4},
    {"id": 6, "товар": "Кавоварка", "місто": " Львів", "ціна": 720, "кількість": 4}, # Повний дублікат рядка id=6
    {"id": 7, "товар": "Кавоварка", "місто": "Львів", "ціна": 720, "кількість": 4}
])

df = raw_orders.copy()
print("Початковий брудний DataFrame (8 рядків):")
display(df)
print("\nІнформація про типи даних та пропуски (df.info()):")
df.info()

Початковий брудний DataFrame (8 рядків):


,id,товар,місто,ціна,кількість
0,1,Кавоварка,Львів,720,4.0
1,2,Кавоварка,Львів,720 грн,NaN
2,3,Кавоварка,ЛЬВІВ,720,4.0
3,4,Кавоварка,Львів,720 грн,NaN
4,5,Кавоварка,Lviv,7200,4.0
5,6,Кавоварка,Львів,720,4.0
6,6,Кавоварка,Львів,720,4.0
7,7,Кавоварка,Львів,720,4.0



Інформація про типи даних та пропуски (df.info()):
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         8 non-null      int64  
 1   товар      8 non-null      str    
 2   місто      8 non-null      str    
 3   ціна       8 non-null      object 
 4   кількість  6 non-null      float64
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 452.0+ bytes


---
## Завдання 2. Пропуски
1. Визначити кількість пропусків у кожному стовпці.
2. Заповнити пропуски в стовпці `"кількість"` медіаною.
3. Письмово пояснити, чому медіана тут доречніша за середнє (з урахуванням розміру вибірки в 8 спостережень).

In [3]:
# 1. Кількість пропусків у кожному стовпці
missing_counts = df.isna().sum()
print("Кількість пропусків у кожному стовпці:")
print(missing_counts)

# 2. Розрахунок медіани та заповнення пропусків
median_qty = df["кількість"].median()
print(f"\nРозрахована медіана для стовпця 'кількість': {median_qty}")

df["кількість"] = df["кількість"].fillna(median_qty)

print("\nСтовпець 'кількість' після заповнення пропусків:")
display(df[["id", "товар", "кількість"]])
print("Перевірка відсутності пропусків у 'кількість':", df["кількість"].isna().sum())

Кількість пропусків у кожному стовпці:
id           0
товар        0
місто        0
ціна         0
кількість    2
dtype: int64

Розрахована медіана для стовпця 'кількість': 4.0

Стовпець 'кількість' після заповнення пропусків:


,id,товар,кількість
0,1,Кавоварка,4.0
1,2,Кавоварка,4.0
2,3,Кавоварка,4.0
3,4,Кавоварка,4.0
4,5,Кавоварка,4.0
5,6,Кавоварка,4.0
6,6,Кавоварка,4.0
7,7,Кавоварка,4.0


Перевірка відсутності пропусків у 'кількість': 0


### Письмове обґрунтування вибору медіани замість середнього:
1. **Стійкість до малих вибірок та викидів (робастність):** У наборі всього 8 спостережень (з яких лише 6 заповнених). На таких малих вибірках середнє арифметичне є вкрай нестабільним: будь-яке одне нетипове замовлення здатне суттєво змістити середнє значення, тоді як медіана відсікає вплив крайніх значень і надійно відображає типову покупку.
2. **Дискретність та фізичний зміст ознаки:** Кількість замовлених одиниць товару є дискретною цілочисельною величиною (неможливо замовити дробову частину кавоварки). Середнє арифметичне практично завжди дає неціле число (наприклад, 4.17 або 3.83), що не має фізичного сенсу в контексті замовлень магазину. Медіана зберігає ціле значення (4.0).
3. **Збереження структури розподілу:** Для товару даного варіанта типовий обсяг покупки становить 4 одиниці. Заповнення медіаною не вносить штучних спотворень у розподіл замовлень магазину.

---
## Завдання 3. Дублікати
1. Знайти дублікати без `subset` (повний збіг рядка).
2. Знайти дублікати з `subset`, що визначає замовлення унікальним (за полем `id` або комбінацією полів).
3. Видалити дублікат і зафіксувати, скільки рядків залишилось у наборі даних.

In [4]:
# 1. Пошук дублікатів без subset (повний збіг усіх стовпців)
duplicates_full = df.duplicated()
print(f"Кількість повних дублікатів (без subset): {duplicates_full.sum()}")
print("Знайдені повні дублікати:")
display(df[df.duplicated(keep=False)])

# 2. Пошук дублікатів із subset=['id'] (унікальний ідентифікатор замовлення)
duplicates_id = df.duplicated(subset=["id"])
print(f"\nКількість дублікатів за subset=['id']: {duplicates_id.sum()}")

# Для порівняння: дублікати за ознаками товару та ціни
duplicates_product_price = df.duplicated(subset=["товар", "ціна"])
print(f"Кількість дублікатів за subset=['товар', 'ціна'] (повторювані комбінації): {duplicates_product_price.sum()}")

# 3. Видалення дублікатів
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)

print(f"\nКількість рядків до видалення: {rows_before}")
print(f"Кількість рядків після видалення дублікатів: {rows_after}")
print(f"Видалено рядків: {rows_before - rows_after}")
display(df)

Кількість повних дублікатів (без subset): 1
Знайдені повні дублікати:


,id,товар,місто,ціна,кількість
5,6,Кавоварка,Львів,720,4.0
6,6,Кавоварка,Львів,720,4.0



Кількість дублікатів за subset=['id']: 1
Кількість дублікатів за subset=['товар', 'ціна'] (повторювані комбінації): 5

Кількість рядків до видалення: 8
Кількість рядків після видалення дублікатів: 7
Видалено рядків: 1


,id,товар,місто,ціна,кількість
0,1,Кавоварка,Львів,720,4.0
1,2,Кавоварка,Львів,720 грн,4.0
2,3,Кавоварка,ЛЬВІВ,720,4.0
3,4,Кавоварка,Львів,720 грн,4.0
4,5,Кавоварка,Lviv,7200,4.0
5,6,Кавоварка,Львів,720,4.0
6,7,Кавоварка,Львів,720,4.0


---
## Завдання 4. Типи даних і узгодження категорій
1. Привести стовпець `"ціна"` до числового типу (`float`), прибравши текстовий суфікс `" грн"`.
2. Привести всі варіанти назви міста до одного канонічного запису (`str.strip()`, `replace()` та приведення регістру).
3. Перевірити результат уніфікації через метод `.unique()`.

In [5]:
# 1. Очищення стовпця 'ціна' та приведення до типу float
print("Тип стовпця 'ціна' до перетворення:", df["ціна"].dtype)
df["ціна"] = df["ціна"].astype(str).str.replace(" грн", "", regex=False).str.strip().astype(float)
print("Тип стовпця 'ціна' після перетворення:", df["ціна"].dtype)

# 2. Узгодження категоріальної назви міста
print("\nВаріанти назви міста до уніфікації:")
print(df["місто"].unique())

# Усунення пробілів, транслітерації та приведення регістру
df["місто"] = (
    df["місто"]
    .astype(str)
    .str.strip()
    .replace({"Lviv": "Львів", "львів": "Львів", "ЛЬВІВ": "Львів"})
    .str.capitalize()
)

# 3. Перевірка результату через .unique()
unique_cities = df["місто"].unique()
print("\nУнікальні значення назви міста після чищення:")
print(unique_cities)

print("\nПоточний стан датасету після Завдання 4:")
display(df)
df.info()

Тип стовпця 'ціна' до перетворення: object
Тип стовпця 'ціна' після перетворення: float64

Варіанти назви міста до уніфікації:
<StringArray>
['Львів', ' Львів ', 'ЛЬВІВ', 'Lviv', ' Львів']
Length: 5, dtype: str

Унікальні значення назви міста після чищення:
<StringArray>
['Львів']
Length: 1, dtype: str

Поточний стан датасету після Завдання 4:


,id,товар,місто,ціна,кількість
0,1,Кавоварка,Львів,720.0,4.0
1,2,Кавоварка,Львів,720.0,4.0
2,3,Кавоварка,Львів,720.0,4.0
3,4,Кавоварка,Львів,720.0,4.0
4,5,Кавоварка,Львів,7200.0,4.0
5,6,Кавоварка,Львів,720.0,4.0
6,7,Кавоварка,Львів,720.0,4.0


<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         7 non-null      int64  
 1   товар      7 non-null      str    
 2   місто      7 non-null      str    
 3   ціна       7 non-null      float64
 4   кількість  7 non-null      float64
dtypes: float64(2), int64(1), str(2)
memory usage: 412.0 bytes


---
## Завдання 5. Викиди
1. Знайти межі IQR (міжквартильного розмаху) для стовпця `"ціна"` очищеного набору.
2. Визначити, чи потрапляє велике значення (7200 грн) у діапазон викидів.
3. Письмово пояснити, яке з двох рішень обрати — вважати це помилкою вводу (зайвий нуль) чи реальним великим замовленням — і чому саме таке рішення.

In [6]:
# 1. Розрахунок квартилів та меж IQR
q1 = df["ціна"].quantile(0.25)
q3 = df["ціна"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"25-й перцентиль (Q1): {q1:.2f} грн")
print(f"75-й перцентиль (Q3): {q3:.2f} грн")
print(f"Міжквартильний розмах (IQR): {iqr:.2f} грн")
print(f"Нижня межа допустимих значень (Q1 - 1.5*IQR): {lower_bound:.2f} грн")
print(f"Верхня межа допустимих значень (Q3 + 1.5*IQR): {upper_bound:.2f} грн")

# 2. Пошук викидів
outliers = df[(df["ціна"] < lower_bound) | (df["ціна"] > upper_bound)]
print(f"\nКількість виявлених викидів за критерієм IQR: {len(outliers)}")
print("Рядки з викидами:")
display(outliers)

# Демонстрація коригування помилки вводу (7200 -> 720 грн)
df_corrected = df.copy()
df_corrected.loc[df_corrected["ціна"] > upper_bound, "ціна"] = 720.0

print("\nДатасет після коригування ціни (фінальний вигляд):")
display(df_corrected)

25-й перцентиль (Q1): 720.00 грн
75-й перцентиль (Q3): 720.00 грн
Міжквартильний розмах (IQR): 0.00 грн
Нижня межа допустимих значень (Q1 - 1.5*IQR): 720.00 грн
Верхня межа допустимих значень (Q3 + 1.5*IQR): 720.00 грн

Кількість виявлених викидів за критерієм IQR: 1
Рядки з викидами:


,id,товар,місто,ціна,кількість
4,5,Кавоварка,Львів,7200.0,4.0



Датасет після коригування ціни (фінальний вигляд):


,id,товар,місто,ціна,кількість
0,1,Кавоварка,Львів,720.0,4.0
1,2,Кавоварка,Львів,720.0,4.0
2,3,Кавоварка,Львів,720.0,4.0
3,4,Кавоварка,Львів,720.0,4.0
4,5,Кавоварка,Львів,720.0,4.0
5,6,Кавоварка,Львів,720.0,4.0
6,7,Кавоварка,Львів,720.0,4.0


### Письмове обґрунтування рішення щодо аномального значення:
Для значення ціни **7200 грн** обрано рішення: **вважати це помилкою вводу (зайвий нуль при введенні оператором: 720 → 7200)** і **скоригувати його до базової ціни 720 грн**, а не вважати реальним великим замовленням або видаляти весь рядок.

**Аргументація:**
1. **Семантика стовпця «ціна»:** Стовпець відображає вартість за одиницю товару в каталозі. За умовою завдання інтернет-магазин продає *один товар* у м. Львів за фіксованою базовою ціною 720 грн. Одинична вартість товару не може раптово зрости в 10 разів для одного випадкового замовлення без зміни специфікації чи комплектації товару.
2. **Чому це не «реальне велике замовлення»:** Якщо клієнт оформлює велике (гуртове) замовлення, то великим має бути значення у стовпці **«кількість»** (наприклад, замовлення 10, 50 чи 100 одиниць товару), а ціна за одиницю залишається базовою або навіть зменшується за рахунок оптової знижки. У нашому випадку кількість становить 4 одиниці, а ціна за одиницю вказана як 7200 грн. Це пряма ознака механічної опечатки (додано зайвий нуль).
3. **Чому коригування краще за видалення:** Повне видалення запису призвело б до втрати інформації про реальний факт продажу та активність клієнта у м. Львів. Коригування значення до 720.0 грн відновлює цілісність даних без викривлення бізнес-метрик.

---
## Відповіді на контрольні запитання

### 1. Чому заповнення пропуску середнім (а не медіаною) може згодом спотворити стандартне відхилення, яке рахуватимемо в наступних лекціях?
- **Штучне заниження вибіркової дисперсії та стандартного відхилення:**
  Формула незміщеної вибіркової дисперсії має вигляд:
  $$s^2 = \frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \bar{x})^2$$
  Коли пропущені значення заповнюються вибірковим середнім $\bar{x}$, кожне таке імп'ютоване спостереження має нульове відхилення від середнього:
  $$(x_{\text{imp}} - \bar{x}) = (\bar{x} - \bar{x}) = 0$$
  В результаті сума квадратів відхилень у чисельнику залишається незмінною, а знаменник $(n - 1)$ збільшується за рахунок зростання кількості заповнених спостережень $n$. Це веде до **штучного стягування розподілу до центру та систематичного заниження стандартного відхилення $s = \sqrt{s^2}$**. На практиці це створює оманливе враження високої точності: довірчі інтервали штучно звужуються, а статистичні тести (t-критерій, ANOVA) починають давати хибнопозитивні результати (помилка I роду).
- **Чутливість середнього на малих вибірках:**
  Середнє арифметичне $\bar{x}$ є неробастною статистикою, яка зазнає значного зміщення навіть від одного викиду чи асиметрії. Заповнення вибірки таким зміщеним середнім закріплює помилку в даних.
- **Перевага медіани:**
  Медіана є порядковою робастною статистикою. Вона не реагує на екстремальні викиди у хвостах розподілу і для дискретних показників (як кількість замовленого товару) зберігає реалістичне ціле значення типового замовлення.

---

### 2. Чи розумієте різницю між `duplicated()` без `subset` і з `subset` — чому результат може відрізнятись?
- **`duplicated()` без `subset` (за замовчуванням `subset=None`):**
  Перевіряє повний посимвольний збіг за **всіма стовпцями** DataFrame. Запис вважається дублікатом лише у випадку, якщо абсолютно кожне значення у всіх стовпцях збігається з попереднім рядком.
- **`duplicated(subset=[...])`:**
  Перевіряє унікальність рядка виключно за **вказаним переліком ключових колонок**, ігноруючи відмінності в інших стовпцях.
- **Чому результати відрізняються:**
  1. **Приховані дублікати замовлення через неузгодженість даних:** Якщо в базу помилково внесли двічі одне й те саме замовлення (однаковий `id=6`), але в першому записі місто вказано як `"Львів"`, а в другому — як `" Львів "` (з пробілами) або іншим регістром:
     - `duplicated()` без `subset` поверне `False` (не побачить дублікат, бо рядки символьно відрізняються);
     - `duplicated(subset=['id'])` поверне `True` і надійно виявить дублювання транзакції за її первинним ключем.
  2. **Збіг неключових параметрів:** Якщо застосувати `subset=['товар', 'місто']`, метод позначить майже всі рядки як дублікати, оскільки товар і місто однакові для багатьох клієнтів. Без `subset` вони визнаються унікальними, оскільки мають різні унікальні `id`.

---

### 3. Чому автоматичне видалення всіх значень поза межами IQR не завжди правильне рішення?
Правило міжквартильного розмаху Тьюкі ($[Q_1 - 1.5 \cdot IQR, \; Q_3 + 1.5 \cdot IQR]$) є евристичним математичним фільтром для виявлення підозрілих точок, а не прямим наказом для видалення рядків:
1. **Викид може бути реальною бізнес-подією:**
   Значення поза межами IQR часто є реальними важливими транзакціями — наприклад, велике гуртове корпоративне замовлення, закупівля обладнання на організацію або піковий продаж під час акцій. Автоматичне видалення спотворить реальні показники сукупного виторгу підприємства та фінансову аналітику.
2. **Помилки введення підлягають виправленню, а не знищенню даних:**
   Як продемонстровано у Завданні 5, ціна 7200 грн виникла внаслідок банальної механічної помилки (зайвий нуль). Якщо видалити весь рядок, бізнес безповоротно втратить відомості про клієнта та факт купівлі товару. Правильним інженерним рішенням є валідація та виправлення ціни на 720 грн.
3. **Асиметричні та важкохвості розподіли:**
   Критерій $1.5 \cdot IQR$ розрахований на симетричні розподіли, близькі до нормального. Для показників з важкими правими хвостами (доходи, ринкова капіталізація, мережевий трафік, закон Парето) великі значення є закономірною властивістю генеральної сукупності. Їх механічне відсікання руйнує об'єктивність дослідження (truncation bias).

---
## Підсумок виконаної роботи
1. Згідно з Варіантом 2 (м. Львів, базова ціна 720 грн, базова кількість 4) побудовано набір даних із 8 замовлень, у якому змодельовано всі 5 дефектів реальних даних.
2. Послідовно застосовано канонічний пайплайн чищення даних:
   - розраховано кількість пропусків та заповнено їх робастною медіаною;
   - проаналізовано та видалено дублікати за допомогою `duplicated()` (як без `subset`, так і з `subset`);
   - скориговано типи даних (стовпець ціни очищено від текстового суфікса та перетворено на `float`);
   - узгоджено текстові категорії міста Львів за допомогою методів очищення рядків;
   - розраховано міжквартильний розмах IQR для пошуку викидів; аргументовано рішення про помилку вводу та скориговано аномальне значення.
3. Наведено ґрунтовні та вичерпні письмові відповіді на всі три теоретичні запитання практики.